In [3]:
import os 
import json 
import sys 
from deepgram import DeepgramClient 

### **Transcription and chunking**

In [4]:
deepgram_api_key = os.getenv("DEEPGRAM_API_KEY")
if not deepgram_api_key:
    print("Please setup Deepgram API Key")
    sys.exit(1)

# Multi-speaker call center audio (agent + customer discussing phone plan cancellation)
AUDIO_URL = "https://static.deepgram.com/examples/en_NatGen_CallCenter_BethTom_CancelPhonePlan.wav"

# Transcribe the audio file 
deepgram = DeepgramClient(api_key = deepgram_api_key)
response = deepgram.listen.v1.media.transcribe_url(
    url = AUDIO_URL,
    model = "nova-2",
    smart_format=True,
    diarize=True,
    utterances=True
)

# Chunk Representation
chunks = []
current_chunk_text = []
current_chunk_meta = {"start": None, "end": None, "speakers": set()}
WORD_LIMIT = 512

# Extract utterances and arrange into chunks
utterances = response.results.utterances
for u in utterances:
    speaker = f"Speaker {int(u.speaker)}"
    text = u.transcript
    timestamp = u.start
    
    # Initialize start time for the new chunk if needed
    if current_chunk_meta["start"] is None:
        current_chunk_meta["start"] = timestamp
        
    # Add to current buffer
    current_chunk_text.append(f"{speaker}: {text}")
    current_chunk_meta["speakers"].add(speaker)
    current_chunk_meta["end"] = u.end  # Update end time continuously
    
    # Check simple length limit (rough word count approximation)
    current_word_count = sum(len(s.split()) for s in current_chunk_text)
    
    if current_word_count >= WORD_LIMIT:
        chunks.append({
            "content": "\n".join(current_chunk_text),
            "metadata": {
                "start_time": current_chunk_meta["start"],
                "end_time": current_chunk_meta["end"],
                "speakers": list(current_chunk_meta["speakers"])
            }
        })
        
        current_chunk_text = []
        current_chunk_meta = {"start": None, "end": None, "speakers": set()}

# Don't forget the last chunk if it has content
if current_chunk_text:
    chunks.append({
        "content": "\n".join(current_chunk_text),
        "metadata": {
            "start_time": current_chunk_meta["start"],
            "end_time": current_chunk_meta["end"],
            "speakers": list(current_chunk_meta["speakers"])
        }
    })

print(json.dumps(chunks, indent=2))

[
  {
    "content": "Speaker 0: Hi. Thank you for calling Premier phone services. This call may be recorded for quality and training purposes.\nSpeaker 0: My name is Tom, and I'll be assisting you. How are you today?\nSpeaker 1: I'm good. Thank you.\nSpeaker 0: Alright. That's good. Can I please have your name?\nSpeaker 1: My name is Beth.\nSpeaker 0: Okay. And, Beth, what is your last name?\nSpeaker 1: Idle.\nSpeaker 0: Could you spell that for me?\nSpeaker 1: Yeah. It's just like it sounds, I d l e. Okay.\nSpeaker 0: I'm not showing a Beth Idol in our system, but there is an Elizabeth.\nSpeaker 1: Oh, yeah. That's my full first name. Yeah. That's me.\nSpeaker 0: Okay, miss Idol. What can I do for you today?\nSpeaker 1: I need some help with my phone plan.\nSpeaker 0: Sure. I can happily help you with that. Can you tell me what plan you currently have?\nSpeaker 1: I have the platinum plan.\nSpeaker 0: Okay. Platinum.\nSpeaker 0: And how many people do you have on your plan right now?